# Earthquake Estimator · Análisis exploratorio de datos

Este cuaderno explora los registros sísmicos del catálogo USGS publicado en Kaggle. Antes de entrenar un modelo, queremos entender qué eventos contiene el archivo, qué información está disponible y qué limitaciones debemos tener en cuenta.

**Pregunta del proyecto:** ¿Podemos estimar la magnitud reportada de un sismo ya detectado a partir de su latitud, longitud y profundidad, y mejorar frente a usar la mediana histórica?

### Fuente y ejecución del cuaderno

Los datos proceden del catálogo **Significant Earthquakes, 1965–2016**, publicado por USGS en Kaggle.

[Consultar el dataset](https://www.kaggle.com/datasets/usgs/earthquake-database)

La configuración del entorno se explica en el [README del proyecto](../README.md). Para ejecutar este cuaderno, guarda `database.csv` en `data/database.csv`, selecciona el Python del entorno `.venv` y ejecuta todas las celdas desde el inicio.

Este notebook documenta los controles de calidad, las transformaciones, las visualizaciones y las decisiones que orientan el modelado.

## 1. Carga y primera inspección del dataset

Cargamos `data/database.csv` con pandas. El archivo contiene **23.412 registros y 21 columnas**.

Usamos `df.head()` para ver las primeras cinco filas y reconocer cómo se registran la fecha, la hora, la ubicación, la profundidad y la magnitud. En esa muestra aparecen valores `NaN`, que representan datos faltantes. Esto nos lleva a revisar cuántos faltantes hay en cada columna del archivo completo.

In [ ]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

df = pd.read_csv(project_root / "data" / "database.csv")
df.head()

## 2. Conteo de valores faltantes por columna

Contamos cuántos valores faltan en cada columna del catálogo completo y ordenamos el resultado de mayor a menor. Así podemos ver dónde se concentra la información incompleta y comprobar si afecta a las variables que queremos usar para estimar la magnitud.

Este conteo es un diagnóstico inicial. Nos ayuda a decidir qué columnas revisar antes de preparar los datos para el modelo.

In [ ]:
faltantes = df.isna().sum().sort_values(ascending=False)
print(faltantes.to_string())

### Hallazgos del conteo de valores faltantes

- `Magnitude`, la variable que queremos estimar, tiene **0 valores faltantes**.
- `Latitude`, `Longitude` y `Depth`, las entradas propuestas para el modelo inicial, también tienen **0 valores faltantes**. Más adelante revisaremos si sus valores están dentro de rangos razonables.
- `Magnitude Error` y `Horizontal Error` concentran la mayor cantidad de faltantes, con **23.085** y **22.256**, respectivamente.

El siguiente paso es convertir estos conteos en porcentajes para saber qué proporción de los 23.412 registros está incompleta en cada columna.

## 3. Porcentaje de valores faltantes por columna

Dividimos el número de valores faltantes de cada columna entre los 23.412 registros del catálogo y multiplicamos por 100. Así podemos ver qué proporción de los registros está incompleta en cada columna.

Por ejemplo, los 23.085 faltantes de `Magnitude Error` representan el **98,60 %** del catálogo. Este resultado nos ayuda a decidir qué columnas requieren una revisión antes de usarlas.

In [ ]:
porcentaje_faltantes = faltantes / len(df) * 100
print(porcentaje_faltantes.round(2).to_string())

### Interpretación de los porcentajes de valores faltantes

- Cinco columnas superan el **80 % de valores faltantes**. Entre ellas, `Magnitude Error`, `Horizontal Error` y `Horizontal Distance` superan el **90 %**. Si las usáramos como entradas, tendríamos que rellenar la mayoría de sus valores o trabajar con muchos menos registros.
- Las entradas propuestas para el modelo inicial (`Latitude`, `Longitude` y `Depth`) y la variable objetivo (`Magnitude`) tienen **0 % de valores faltantes**. Podemos conservarlas para el análisis sin imputar datos en esas columnas. Aún debemos revisar si sus valores están dentro de rangos razonables.
- `Magnitude Type` tiene solo **3 valores faltantes**, equivalentes al **0,01 %**. Podemos usar esta columna para describir los tipos de magnitud reportados en el catálogo.

Por ahora, el CSV se conserva completo. Estos resultados documentan la disponibilidad de los datos y orientan la selección inicial de variables.

## 4. Tipos de eventos del catálogo

Revisamos la columna `Type` para identificar qué tipos de eventos contiene el catálogo y cuántos registros hay de cada uno.

Como el proyecto se enfoca en terremotos, este conteo nos permitirá saber cuántas filas corresponden a `Earthquake` y cuántas quedarán fuera del análisis cuando apliquemos el filtro. Primero contamos los eventos de cada tipo para documentar el criterio de selección.

In [ ]:
conteo_eventos = df["Type"].value_counts()
print(conteo_eventos.to_string())

### Hallazgos por tipo de evento

De los **23.412 registros** del catálogo, **23.232** están clasificados como `Earthquake` y representan el **99,23 %** del total. Los **180 registros restantes** corresponden a 175 explosiones nucleares, 4 explosiones y 1 evento clasificado como `Rock Burst`.

Como el objetivo del proyecto es estimar la magnitud de terremotos, en el siguiente paso seleccionaremos las filas con `Type == "Earthquake"`. Trabajaremos con esa selección en una tabla separada para conservar el catálogo completo como referencia.

In [ ]:
terremotos = df.loc[df["Type"] == "Earthquake"].copy()

print(f"Registros del catálogo: {len(df)}")
print(f"Terremotos seleccionados: {len(terremotos)}")
print(f"Otros eventos excluidos: {len(df) - len(terremotos)}")


### Resultado del filtro y alcance del análisis

El catálogo original contiene **23.412 registros**. Al seleccionar los eventos clasificados como `Earthquake`, obtuvimos **23.232 terremotos** y excluimos **180 eventos** de otros tipos.

Conservamos el catálogo original en `df` y guardamos la selección en `terremotos`. Esta segunda tabla será la base para explorar la distribución de magnitudes, la ubicación de los epicentros y la relación entre profundidad y magnitud.

El diagnóstico de valores faltantes presentado arriba corresponde al catálogo completo. A partir de este punto, los resultados del análisis se referirán a los terremotos seleccionados.

## 5. Tipos de variables y controles de calidad

A partir de esta sección trabajamos con `terremotos`, la tabla de **23.232 eventos** seleccionados del catálogo.

Primero revisamos qué tipo de dato asignó pandas a cada columna. Esto nos permite distinguir las variables que podemos resumir con estadísticas numéricas, las que describen categorías y las que requieren un tratamiento especial, como las fechas y los identificadores.

`Magnitude` es la variable que queremos estimar. `Latitude`, `Longitude` y `Depth` son las entradas propuestas para el modelo inicial. La revisión de tipos nos permite comprobar cómo se cargaron estas columnas antes de validar sus rangos y analizar sus distribuciones.

In [ ]:
tipos_detectados = terremotos.dtypes.rename("Tipo detectado por pandas")
print(tipos_detectados.to_string())

### Interpretación de los tipos de datos

Pandas cargó **12 columnas numéricas** (`float64`) y **9 columnas de texto** (`str`). Para el análisis, también clasificamos las columnas según lo que representan:

| Tipo de variable | Ejemplos | Tratamiento en el EDA |
|---|---|---|
| Mediciones numéricas | `Magnitude`, `Depth`, `Latitude`, `Longitude` | Revisar rangos y explorar sus distribuciones o su ubicación en el mapa, según corresponda. |
| Conteos discretos | `Depth Seismic Stations`, `Magnitude Seismic Stations` | Interpretarlos como cantidades enteras de estaciones. Se cargaron como `float64` porque contienen valores faltantes. |
| Categóricas | `Type`, `Magnitude Type`, `Source`, `Location Source`, `Magnitude Source`, `Status` | Resumirlas mediante frecuencias. Después del filtro, `Type` solo contiene la categoría `Earthquake`. |
| Temporales | `Date`, `Time` | Revisar sus formatos y convertirlas para poder ordenar los eventos y extraer componentes como el año. |
| Identificador | `ID` | Usarlo para comprobar si hay identificadores repetidos. |

`Magnitude` es la variable objetivo. `Latitude`, `Longitude` y `Depth` son las entradas propuestas para el modelo inicial. Las cuatro se cargaron como variables numéricas. El siguiente control de calidad será verificar si hay registros duplicados o identificadores repetidos.

### Revisión de registros duplicados e identificadores

En la tabla `terremotos` hacemos dos controles de calidad. Primero, contamos las filas idénticas para detectar si algún registro completo aparece más de una vez. Después, revisamos si se repite algún `ID` entre los registros que tienen identificador.

Estas comprobaciones son distintas. Dos filas pueden compartir el mismo `ID` y tener valores diferentes en otras columnas. Si encontramos alguna repetición, revisaremos los registros antes de decidir cómo tratarla.

In [ ]:
filas_duplicadas = terremotos.duplicated().sum()

ids = terremotos["ID"]
ids_faltantes = ids.isna().sum()
ids_duplicados = ids.dropna().duplicated().sum()

print(f"Filas completas duplicadas: {filas_duplicadas}")
print(f"IDs faltantes: {ids_faltantes}")
print(f"Repeticiones de ID después de la primera aparición: {ids_duplicados}")

### Hallazgos de la revisión de duplicados

Entre los **23.232 terremotos seleccionados** encontramos **0 filas completas duplicadas**, **0 identificadores faltantes** y **0 identificadores repetidos**. Cada fila tiene un `ID` distinto y podemos conservar todos los registros en esta etapa.

El siguiente control será revisar si la latitud, la longitud, la profundidad y la magnitud tienen valores dentro de los rangos esperados.

### Validación de rangos de las variables principales

Revisamos el mínimo y el máximo de `Latitude`, `Longitude`, `Depth` y `Magnitude` en los 23.232 terremotos seleccionados.

Comprobamos que la latitud esté entre −90° y 90° y la longitud entre −180° y 180°. También verificamos si hay magnitudes por debajo de 5,5, el umbral reportado para este catálogo. Para la profundidad, mostramos el rango observado y contamos los valores negativos para revisarlos por separado.

Esta revisión permite detectar registros que requieren inspección antes de analizar las distribuciones. Conservamos todos los registros durante el diagnóstico.

In [ ]:
columnas = ["Latitude", "Longitude", "Depth", "Magnitude"]
rangos = terremotos[columnas].agg(["min", "max"]).T
print(rangos.to_string())

latitud_fuera = (~terremotos["Latitude"].between(-90, 90)).sum()
longitud_fuera = (~terremotos["Longitude"].between(-180, 180)).sum()
profundidades_negativas = terremotos["Depth"].lt(0).sum()
magnitudes_bajo_umbral = terremotos["Magnitude"].lt(5.5).sum()

print(f"\nLatitudes fuera de rango: {latitud_fuera}")
print(f"Longitudes fuera de rango: {longitud_fuera}")
print(f"Profundidades negativas para revisar: {profundidades_negativas}")
print(f"Magnitudes por debajo de 5,5: {magnitudes_bajo_umbral}")

### Hallazgos de la validación de rangos

| Variable | Rango observado | Resultado del control |
|---|---|---|
| `Latitude` | −77,080° a 86,005° | Las 23.232 latitudes están dentro de los límites geográficos. |
| `Longitude` | −179,997° a 179,998° | Las 23.232 longitudes están dentro de los límites geográficos. |
| `Depth` | −1,1 a 700 km | Se identificaron 3 profundidades negativas para revisión individual. |
| `Magnitude` | 5,5 a 9,1 | Todos los registros cumplen el umbral mínimo de 5,5 del catálogo. |

Los tres registros con profundidad negativa se conservan durante el diagnóstico. Antes de decidir cómo tratarlos, revisaremos su identificador, su profundidad y la fuente de localización. Los máximos de 700 km y magnitud 9,1 se examinarán al estudiar la distribución de cada variable.

### Inspección de profundidades negativas

La validación de rangos identificó tres terremotos con profundidad negativa. Para revisar cada caso, mostramos su identificador, fecha, coordenadas, profundidad, magnitud y fuente de localización.

Ordenamos los registros por profundidad para comparar los valores observados y comprobar si comparten una misma fuente. Durante esta inspección conservamos los valores originales.

In [ ]:
columnas_revision = [
    "ID", "Date", "Latitude", "Longitude",
    "Depth", "Magnitude", "Location Source",
]

profundidades_negativas = (
    terremotos.loc[terremotos["Depth"].lt(0), columnas_revision]
    .sort_values("Depth")
)

print(profundidades_negativas.to_string(index=False))

### Hallazgos de la inspección de profundidades negativas

Encontramos **3 registros con profundidad negativa** entre los **23.232 terremotos** analizados. Representan aproximadamente el **0,013 %** de la selección.

- `CI3043549` y `CI3031111` se registraron el **28 de junio de 1992**. Sus coordenadas son cercanas y ambos tienen `CI` como fuente de localización. Sus profundidades reportadas son **−1,100 km** y **−0,097 km**, respectivamente.
- `NC95459` se registró el **21 de julio de 1986**, tiene `NC` como fuente de localización y una profundidad reportada de **−0,076 km**.

La tabla permite identificar los casos y sus fuentes, pero no explica por qué se reportaron profundidades negativas. Durante el modelado podremos comparar entrenamientos con y sin estos tres registros, manteniendo los mismos conjuntos de validación y prueba.

### Normalización de la columna `Date`

La columna `Date` se cargó como texto y contiene dos formatos. La mayoría de los registros usa mes/día/año, como `06/28/1992`. Otros incluyen la fecha y la hora en formato ISO, como `1975-02-23T02:58:41.000Z`.

Convertimos ambos formatos a una fecha de calendario y guardamos el resultado en `fecha_evento`. Esto permitirá ordenar los terremotos cronológicamente y agruparlos por año. Conservamos `Date` para poder contrastar los resultados con el valor original.

Al final contamos cuántas fechas se pudieron interpretar y revisamos el periodo que cubren.

In [ ]:
fechas_originales = terremotos["Date"].str.strip()

fechas_mes_dia_anio = pd.to_datetime(
    fechas_originales, format="%m/%d/%Y", errors="coerce"
)

fechas_iso = pd.to_datetime(
    fechas_originales.where(fechas_mes_dia_anio.isna()),
    format="ISO8601",
    errors="coerce",
    utc=True,
)

terremotos["fecha_evento"] = fechas_mes_dia_anio.fillna(
    fechas_iso.dt.normalize().dt.tz_localize(None)
)

print(f"Fechas interpretadas con mes/día/año: {fechas_mes_dia_anio.notna().sum()}")
print(f"Fechas interpretadas con formato ISO: {fechas_iso.notna().sum()}")
print(f"Fechas sin interpretar: {terremotos['fecha_evento'].isna().sum()}")
print(f"Primera fecha: {terremotos['fecha_evento'].min():%Y-%m-%d}")
print(f"Última fecha: {terremotos['fecha_evento'].max():%Y-%m-%d}")

### Hallazgos de la normalización de fechas

De los **23.232 terremotos** seleccionados, **23.229** tenían fechas en formato mes/día/año y **3** en formato ISO. Los dos formatos se interpretaron correctamente y quedaron **0 fechas sin convertir**.

El periodo observado va del **2 de enero de 1965** al **30 de diciembre de 2016**. La columna `fecha_evento` nos permite ordenar los registros y agruparlos por año. Conservamos `Date` como referencia para consultar el valor original.

### Registros de terremotos por año

Extraemos el año de `fecha_evento` y lo guardamos en una nueva columna llamada `anio_evento`. Esta transformación permite agrupar los terremotos por año y contar cuántos registros hay en cada periodo.

El conteo describe los eventos incluidos en el catálogo, cuyo umbral mínimo de magnitud es 5,5. Las diferencias entre años pueden reflejar cambios en la actividad sísmica o en la cobertura del catálogo.

In [ ]:
terremotos["anio_evento"] = terremotos["fecha_evento"].dt.year

conteo_anual = (
    terremotos.groupby("anio_evento")
    .size()
    .rename("registros")
    .sort_index()
)

print(f"Años con registros: {len(conteo_anual)}")
print(f"Total de registros agrupados: {conteo_anual.sum()}")

print("\nCinco años con más registros:")
print(conteo_anual.nlargest(5).to_string())

print("\nCinco años con menos registros:")
print(conteo_anual.nsmallest(5).to_string())

### Hallazgos del conteo anual

El catálogo contiene **23.232 terremotos** distribuidos en **52 años con registros**, de 1965 a 2016. La suma de los conteos anuales coincide con el total de terremotos seleccionados, lo que confirma que todos quedaron incluidos al agrupar por año.

El año con más registros fue **2011**, con **713 terremotos**, y el año con menos fue **1966**, con **233**. Estos valores muestran que el número de registros varía entre años. Para evaluar si hay una tendencia sostenida, debemos observar la serie completa en orden cronológico y tener en cuenta la cobertura del catálogo.

### Evolución anual de los registros de terremotos

Graficamos los conteos de 1965 a 2016 en orden cronológico. Cada punto representa un año y su altura indica cuántos terremotos de magnitud 5,5 o superior quedaron registrados en el catálogo.

La gráfica permite ver si los años con conteos altos forman parte de un periodo con muchos registros o si son picos frente a los años vecinos. En particular, revisaremos cómo se comporta 2011, el año con el mayor conteo.

In [ ]:
import plotly.express as px

datos_anuales = conteo_anual.reset_index()

fig = px.line(
    datos_anuales,
    x="anio_evento",
    y="registros",
    markers=True,
    title="Terremotos registrados por año (1965–2016)",
    labels={
        "anio_evento": "Año",
        "registros": "Terremotos registrados",
    },
)

fig.update_yaxes(rangemode="tozero")
fig.show()

### Interpretación de la evolución anual

El número de terremotos registrados fluctúa entre 1965 y 2016. El menor conteo fue de **233 registros en 1966** y el mayor fue de **713 en 2011**.

El pico de 2011 se distingue de los años vecinos. El catálogo pasó de **559 registros en 2010** a **713 en 2011**, y el conteo bajó en 2012. La caída posterior muestra que ese máximo no se mantuvo en los años siguientes.

La serie describe los eventos incluidos en el catálogo, con magnitudes reportadas de **5,5 o más**. Para atribuir las variaciones a cambios en la actividad sísmica, habría que revisar si la cobertura del catálogo fue comparable a lo largo del periodo.

## 6. Distribución de la magnitud de los terremotos

Exploramos `Magnitude`, la variable que queremos estimar. El histograma agrupa las magnitudes en intervalos de 0,1 y muestra cuántos terremotos quedaron registrados en cada intervalo. Así podemos ver dónde se concentran los valores y qué tan frecuentes son las magnitudes altas.

Calculamos la media, la mediana y la moda. La media resume el promedio, la mediana marca el centro de los datos ordenados y la moda identifica el valor reportado con mayor frecuencia.

El catálogo incluye eventos con magnitud de 5,5 o superior. Por eso, la distribución describe los terremotos seleccionados dentro de ese rango.

In [ ]:
magnitud = terremotos["Magnitude"]

media_magnitud = magnitud.mean()
mediana_magnitud = magnitud.median()
modas_magnitud = magnitud.mode().tolist()

print(f"Media: {media_magnitud:.2f}")
print(f"Mediana: {mediana_magnitud:.2f}")
print("Moda(s): " + ", ".join(f"{valor:.2f}" for valor in modas_magnitud))

fig_magnitud = px.histogram(
    terremotos,
    x="Magnitude",
    title="Distribución de magnitudes de los terremotos registrados",
    labels={"Magnitude": "Magnitud reportada"},
)

fig_magnitud.update_traces(xbins=dict(start=5.45, end=9.15, size=0.1))
fig_magnitud.update_yaxes(title_text="Número de terremotos")
fig_magnitud.show()

### Interpretación de la distribución de magnitudes

Las magnitudes reportadas se concentran cerca de **5,5**, el umbral mínimo del catálogo. A medida que aumenta la magnitud, los conteos por intervalo generalmente disminuyen y la distribución se extiende hasta **9,1**. Esta forma se describe como **asimetría hacia la derecha**.

La gráfica describe los terremotos incluidos en el catálogo a partir de magnitud 5,5. Para la evaluación del modelo, convendrá revisar por separado el error en los eventos de magnitud **7 o superior**, porque su baja frecuencia puede hacer que tengan poco peso en el error global.

#### Media, mediana y moda

La **media fue 5,88**, la **mediana 5,70** y la **moda 5,50**. La media supera a la mediana en **0,18**, lo cual es coherente con la cola hacia la derecha que observamos en el histograma. Las magnitudes altas son menos frecuentes, pero elevan el promedio.

La mediana de **5,70** indica que al menos la mitad de los terremotos del catálogo tiene una magnitud reportada de **5,70 o menos**. La moda de **5,50** es el valor exacto más frecuente y coincide con el umbral mínimo del catálogo.

La mediana describe el centro de esta distribución y servirá como referencia para el modelo. Al evaluar el modelo, calcularemos esa referencia usando únicamente los datos de entrenamiento.

### Dispersión y valores atípicos de la magnitud

Calculamos el primer cuartil (Q1) y el tercer cuartil (Q3). La diferencia entre ambos es el rango intercuartílico (IQR), que mide la amplitud del 50 % central de las magnitudes registradas.

Usamos un diagrama de caja y la regla de 1,5 veces el IQR para identificar magnitudes alejadas de ese rango central. Los valores señalados se conservan durante la exploración, ya que una magnitud poco frecuente puede corresponder a un terremoto real.

In [ ]:
q1_magnitud = magnitud.quantile(0.25)
q3_magnitud = magnitud.quantile(0.75)
iqr_magnitud = q3_magnitud - q1_magnitud

limite_inferior = q1_magnitud - 1.5 * iqr_magnitud
limite_superior = q3_magnitud + 1.5 * iqr_magnitud

print(f"Q1: {q1_magnitud:.2f}")
print(f"Q3: {q3_magnitud:.2f}")
print(f"IQR: {iqr_magnitud:.2f}")
print(f"Límite inferior: {limite_inferior:.2f}")
print(f"Límite superior: {limite_superior:.2f}")
print(f"Magnitudes por debajo del límite: {magnitud.lt(limite_inferior).sum()}")
print(f"Magnitudes por encima del límite: {magnitud.gt(limite_superior).sum()}")

fig_caja_magnitud = px.box(
    terremotos,
    x="Magnitude",
    points="outliers",
    title="Diagrama de caja de las magnitudes reportadas",
    labels={"Magnitude": "Magnitud reportada"},
)
fig_caja_magnitud.show()

### Interpretación del diagrama de caja

El 50 % central de las magnitudes se ubica entre **5,60 (Q1)** y **6,00 (Q3)**. La diferencia entre ambos cuartiles es de **0,40 unidades de magnitud**. La línea dentro de la caja marca la mediana de **5,70**.

Con la regla de **1,5 veces el IQR**, obtuvimos un límite inferior de **5,00** y uno superior de **6,60**. Ningún registro quedó por debajo del límite inferior, lo cual es coherente con el umbral mínimo de **5,50** del catálogo. Por encima del límite superior quedaron **1.471 terremotos**, equivalentes al **6,33 %** de los 23.232 analizados.

El bigote derecho más largo y los puntos que se extienden hasta **9,10** son coherentes con la asimetría hacia la derecha observada en el histograma. Conservamos esos registros porque la regla del IQR señala valores alejados del centro de la distribución y, por sí sola, no indica errores de calidad.

## 7. Distribución de la profundidad de los terremotos

Analizamos la profundidad reportada en kilómetros para entender en qué intervalos se concentran los terremotos del catálogo y cómo se distribuyen los eventos más profundos.

Calculamos la media, la mediana y los cuartiles, y usamos un histograma para comparar las frecuencias por intervalo. Conservamos los tres registros con profundidad negativa que identificamos durante la validación de rangos.

In [ ]:
profundidad = terremotos["Depth"]

print(f"Media: {profundidad.mean():.2f} km")
print(f"Mediana: {profundidad.median():.2f} km")
print(f"Q1: {profundidad.quantile(0.25):.2f} km")
print(f"Q3: {profundidad.quantile(0.75):.2f} km")

fig_profundidad = px.histogram(
    terremotos,
    x="Depth",
    title="Distribución de las profundidades reportadas",
    labels={"Depth": "Profundidad reportada (km)"},
)

fig_profundidad.update_traces(xbins=dict(start=-10, end=710, size=10))
fig_profundidad.update_yaxes(title_text="Número de terremotos")
fig_profundidad.show()

### Interpretación de la distribución de profundidades

El histograma muestra que los registros se concentran en profundidades bajas, con los mayores conteos en los intervalos de **10 a 20 km** y de **30 a 40 km**. La distribución tiene una **cola hacia la derecha** que se extiende hasta **700 km**.

La **mediana es de 33,00 km** y la **media es de 71,31 km**. Los terremotos más profundos, aunque menos frecuentes, elevan el promedio. En este catálogo, la mediana representa mejor la profundidad típica.

El primer cuartil (Q1) es **15,00 km** y el tercer cuartil (Q3) es **54,80 km**. Entre esos dos valores se ubica aproximadamente la mitad central de los registros. El rango intercuartílico (IQR) es de **39,80 km**.

## 8. Relación entre profundidad y magnitud

Exploramos si los terremotos más profundos tienden a presentar magnitudes reportadas distintas. En el diagrama de dispersión, cada punto representa un terremoto. El eje horizontal muestra la profundidad en kilómetros y el eje vertical muestra la magnitud reportada.

Complementamos la gráfica con dos coeficientes. Pearson resume la asociación lineal. Spearman compara el orden de los valores y permite ver si, al aumentar la profundidad, la magnitud tiende a subir o bajar aunque la relación no siga una línea recta.

El análisis describe los terremotos registrados en el catálogo, que incluye magnitudes de 5,5 o más.

In [ ]:
variables = terremotos[["Depth", "Magnitude"]]

pearson = variables["Depth"].corr(variables["Magnitude"])
spearman = variables["Depth"].rank().corr(variables["Magnitude"].rank())

print(f"Correlación de Pearson: {pearson:.3f}")
print(f"Correlación de Spearman: {spearman:.3f}")

fig_profundidad_magnitud = px.scatter(
    terremotos,
    x="Depth",
    y="Magnitude",
    opacity=0.2,
    title="Profundidad frente a magnitud de los terremotos registrados",
    labels={
        "Depth": "Profundidad reportada (km)",
        "Magnitude": "Magnitud reportada",
    },
)

fig_profundidad_magnitud.show()

### Interpretación de la relación entre profundidad y magnitud

La mayor concentración de registros se observa en profundidades bajas y magnitudes cercanas al mínimo del catálogo, **5,5**. En la gráfica aparecen magnitudes similares a profundidades muy distintas, y la nube de puntos no muestra una subida o bajada clara a lo largo de todo el rango.

La correlación de **Pearson fue 0,023** y la de **Spearman fue −0,037**. Ambos valores están muy cerca de cero. En el conjunto completo se observa una asociación lineal muy débil y poca evidencia de una tendencia general creciente o decreciente entre profundidad y magnitud.

Conservamos `Depth` como candidata para el modelo inicial. Para saber si aporta información al combinarla con latitud y longitud, compararemos el error de los modelos con y sin profundidad usando la misma partición temporal.

## 9. Distribución geográfica de los epicentros

Representamos los 23.232 terremotos seleccionados usando `Latitude` y `Longitude`. Cada punto muestra la ubicación de un epicentro y su color indica la magnitud reportada.

El mapa permite explorar dónde se concentran los registros y cómo se distribuyen espacialmente las magnitudes. La superposición de puntos puede ocultar eventos, por lo que la imagen se interpreta como una exploración de ubicaciones y concentraciones visibles.

Al pasar el cursor por un punto, podemos consultar su identificador, fecha, magnitud y profundidad.

In [ ]:
fig_mapa = px.scatter_geo(
    terremotos,
    lat="Latitude",
    lon="Longitude",
    color="Magnitude",
    hover_name="ID",
    hover_data={
        "fecha_evento": True,
        "Depth": ":.2f",
        "Magnitude": ":.2f",
        "Latitude": ":.3f",
        "Longitude": ":.3f",
    },
    color_continuous_scale="Viridis",
    projection="natural earth",
    opacity=0.5,
    title="Epicentros de los terremotos registrados (1965–2016)",
    labels={
        "Magnitude": "Magnitud reportada",
        "Depth": "Profundidad (km)",
        "fecha_evento": "Fecha",
        "Latitude": "Latitud",
        "Longitude": "Longitud",
    },
)

fig_mapa.update_traces(marker=dict(size=3))
fig_mapa.update_geos(
    showland=True,
    landcolor="lightgray",
    showocean=True,
    oceancolor="aliceblue",
)
fig_mapa.update_layout(height=600)
fig_mapa.show()

### Interpretación de la distribución geográfica

Los epicentros forman franjas y agrupaciones visibles. Destacan las concentraciones a lo largo del borde occidental de América, en Alaska y alrededor de Japón, Filipinas, Indonesia y Nueva Zelanda. También se observan alineaciones de registros en el océano Atlántico y en el océano Índico.

Predominan los puntos de color morado, correspondientes a magnitudes cercanas al mínimo del catálogo. Esto es coherente con el histograma de magnitudes. La superposición de puntos dificulta distinguir algunos eventos y comparar visualmente la cantidad exacta de registros entre regiones.

El mapa muestra una distribución espacial heterogénea. Conservamos latitud y longitud como entradas candidatas para el modelo y comprobaremos su aporte mediante evaluación temporal. Las concentraciones observadas describen este catálogo histórico de eventos con magnitud de 5,5 o más.

## 10. Tipos de magnitud reportados

Revisamos `Magnitude Type` para identificar qué tipos de magnitud reúne el catálogo y qué proporción representa cada uno entre los 23.232 terremotos seleccionados.

Calculamos frecuencias y porcentajes e incluimos los registros sin tipo de magnitud. Esta revisión permitirá documentar la composición de nuestra variable objetivo, `Magnitude`, y orientar el análisis de resultados por tipo de magnitud.

In [ ]:
conteo_tipos_magnitud = (
    terremotos["Magnitude Type"]
    .fillna("Sin información")
    .value_counts()
)

resumen_tipos_magnitud = conteo_tipos_magnitud.to_frame(name="registros")
resumen_tipos_magnitud["porcentaje"] = (
    resumen_tipos_magnitud["registros"] / len(terremotos) * 100
).round(2)

print(resumen_tipos_magnitud.to_string())
print(f"\nTotal de registros: {resumen_tipos_magnitud['registros'].sum()}")

### Hallazgos sobre los tipos de magnitud

Los tipos más frecuentes son **MW**, con **7.722 registros (33,24 %)**, y **MWC**, con **5.668 (24,40 %)**. Juntos representan el **57,64 %** de los terremotos seleccionados.

También tienen una participación importante `MB` (15,47 %), `MWB` (10,58 %), `MWW` (8,54 %) y `MS` (7,33 %). Hay **3 registros sin información**, equivalentes al **0,01 %**.

La suma de las categorías coincide con los **23.232 terremotos** analizados. La variable objetivo `Magnitude` reúne valores reportados con distintos tipos de magnitud. Esta heterogeneidad se documenta como una limitación para interpretar los resultados del modelo.

Conservamos los registros durante el EDA y usaremos `Magnitude Type` para revisar el error por categoría cuando haya suficientes casos. El modelo inicial tendrá como entradas latitud, longitud y profundidad.

## 11. Selección inicial de variables para el modelo

Definimos `Magnitude` como variable objetivo y seleccionamos `Latitude`, `Longitude` y `Depth` como entradas del modelo inicial. Estas tres columnas son numéricas y tienen todos sus valores disponibles en los terremotos seleccionados.

La ubicación muestra concentraciones espaciales en el mapa. La profundidad presenta una asociación global débil con la magnitud, pero se conserva como candidata para comprobar su aporte al combinarla con la ubicación.

`fecha_evento` y `anio_evento` permiten ordenar los registros y preparar una evaluación temporal. `Magnitude Type` se conserva para describir el catálogo y revisar posteriormente el error por categoría.

El modelo estimará la magnitud reportada de un terremoto ya detectado, con ubicación y profundidad disponibles. Su alcance corresponde al catálogo analizado, que incluye magnitudes de 5,5 o superiores.

In [ ]:
variables_entrada = ["Latitude", "Longitude", "Depth"]
variable_objetivo = "Magnitude"

X = terremotos[variables_entrada].copy()
y = terremotos[variable_objetivo].copy()

print(f"Filas y columnas de entrada: {X.shape}")
print(f"Valores de la variable objetivo: {len(y)}")

print("\nFaltantes en las entradas:")
print(X.isna().sum().to_string())

print(f"\nFaltantes en la variable objetivo: {y.isna().sum()}")

### Verificación de las entradas y la variable objetivo

La tabla `X` contiene **23.232 filas y 3 columnas** (`Latitude`, `Longitude` y `Depth`). La serie `y` contiene las **23.232 magnitudes** correspondientes. Ambas se construyeron a partir de las mismas filas de `terremotos`, conservando su correspondencia.

Las entradas y la variable objetivo tienen **0 valores faltantes**, por lo que esta selección no requiere imputación. Se mantienen los tres registros con profundidad negativa, identificados y documentados durante los controles de calidad.

La selección queda preparada para organizar los datos cronológicamente y definir los periodos de entrenamiento, validación y prueba.

## 12. Propuesta de evaluación temporal

Organizamos los terremotos en tres periodos consecutivos para evaluar la capacidad del modelo de generalizar a años posteriores.

| Conjunto | Periodo | Función |
|---|---|---|
| Entrenamiento | 1965–2006 | Ajustar el modelo y cualquier transformación que aprenda de los datos. |
| Validación | 2007–2011 | Comparar modelos y elegir hiperparámetros. |
| Prueba | 2012–2016 | Medir el desempeño final después de fijar las decisiones de modelado. |

La división conserva el orden cronológico. El modelo se ajustará con eventos anteriores y se evaluará con eventos posteriores.

El EDA presentado exploró el catálogo completo, incluidos los años propuestos para prueba. Documentamos este alcance y fijamos ahora los periodos, antes de entrenar modelos. El periodo de prueba ya tuvo exposición descriptiva, por lo que la evaluación final deberá interpretarse con esa limitación.

In [ ]:
periodos = {
    "Entrenamiento": terremotos["anio_evento"].between(1965, 2006),
    "Validación": terremotos["anio_evento"].between(2007, 2011),
    "Prueba": terremotos["anio_evento"].between(2012, 2016),
}

resumen_particiones = pd.DataFrame(
    [
        {
            "conjunto": nombre,
            "primer_anio": terremotos.loc[mascara, "anio_evento"].min(),
            "ultimo_anio": terremotos.loc[mascara, "anio_evento"].max(),
            "registros": int(mascara.sum()),
        }
        for nombre, mascara in periodos.items()
    ]
)

resumen_particiones["porcentaje"] = (
    resumen_particiones["registros"] / len(terremotos) * 100
).round(2)

print(resumen_particiones.to_string(index=False))
print(f"\nTotal de registros asignados: {resumen_particiones['registros'].sum()}")

### Resultado de la división temporal

Asignamos **18.026 terremotos (77,59 %)** a entrenamiento, **2.905 (12,50 %)** a validación y **2.301 (9,90 %)** a prueba.

Los periodos son consecutivos y no se superponen. La suma coincide con los **23.232 terremotos** seleccionados, por lo que todos los registros quedaron asignados. La diferencia de 0,01 puntos porcentuales frente al 100 % se debe al redondeo.

Durante el modelado, cualquier transformación que aprenda parámetros y la mediana del modelo de referencia se ajustarán con los datos de entrenamiento. La validación servirá para elegir la configuración y la prueba para medir el resultado final, teniendo en cuenta la exposición descriptiva de este periodo durante el EDA.

## 13. Decisiones de preparación de variables

| Decisión | Justificación |
|---|---|
| Seleccionar `Type == "Earthquake"` | Delimita el análisis a los eventos que corresponden al objetivo del proyecto. |
| Convertir `Date` en `fecha_evento` | Unifica los dos formatos encontrados y permite ordenar cronológicamente los eventos. |
| Extraer `anio_evento` | Permite resumir los registros por año y definir los periodos de evaluación. |
| Usar latitud, longitud y profundidad como entradas iniciales | Son variables numéricas completas y disponibles en el escenario propuesto de un evento ya localizado. |
| Conservar los eventos señalados por el IQR | Su distancia respecto al centro de la distribución no basta para considerarlos errores. |
| Conservar las tres profundidades negativas durante el EDA | Se documentaron los registros y queda pendiente verificar su interpretación con las fuentes. |
| Mantener `Magnitude Type` para análisis por categoría | Permite documentar la composición del objetivo y revisar diferencias de desempeño. |

Las tres entradas seleccionadas tienen todos sus valores disponibles, por lo que no requieren imputación. El árbol de decisión y el bosque aleatorio propuestos pueden trabajar con estas entradas numéricas sin escalamiento. La codificación de categorías se considerará si se incorporan variables categóricas a una versión posterior del modelo.

## 14. Limitaciones del análisis

- **Cobertura del catálogo.** Analizamos terremotos con magnitud reportada de 5,5 o superior entre 1965 y 2016. Las conclusiones corresponden a este conjunto de eventos. Al interpretar cambios entre años, debemos considerar posibles diferencias en la cobertura del catálogo.

- **Tipos de magnitud.** La variable objetivo reúne valores reportados bajo distintas categorías de `Magnitude Type`. Esta heterogeneidad puede influir en las distribuciones y en los errores del modelo. Las categorías con pocos registros tendrán evaluaciones menos estables.

- **Profundidades pendientes de verificación.** Conservamos tres profundidades negativas y documentamos sus identificadores y fuentes. Queda pendiente verificar su interpretación con las fuentes originales.

- **Representación desigual.** Las magnitudes cercanas a 5,5 son mucho más frecuentes que las magnitudes altas. Además del error global, revisaremos el desempeño en eventos de magnitud 7 o superior e indicaremos cuántos casos respaldan esa evaluación.

- **Dependencia entre eventos.** Los terremotos cercanos en ubicación y fecha pueden estar relacionados. La división temporal permite evaluar años posteriores, aunque puede persistir dependencia espacial y temporal entre los registros.

- **Exposición del periodo de prueba.** El EDA incluyó el catálogo completo. Documentamos esta exposición descriptiva y fijamos los periodos antes del entrenamiento. Las decisiones de ajuste del modelo se tomarán con entrenamiento y validación.

- **Capacidad predictiva pendiente de comprobar.** El EDA identifica patrones y prepara variables candidatas. La utilidad de latitud, longitud y profundidad se determinará comparando los modelos con una referencia basada en la mediana de entrenamiento.

## 15. Conclusiones del EDA y preparación del modelado

1. **Definimos la población de estudio.** De los 23.412 registros originales seleccionamos 23.232 terremotos. La selección tiene identificadores completos y únicos, cero filas duplicadas y datos completos en las tres entradas propuestas y en la magnitud objetivo.

2. **Normalizamos las fechas.** Convertimos correctamente 23.229 fechas en formato mes/día/año y 3 en formato ISO. Creamos `fecha_evento` y `anio_evento` para analizar el periodo 1965–2016 y preparar la evaluación temporal.

3. **La magnitud presenta asimetría hacia la derecha.** La media es 5,88, la mediana 5,70 y la moda 5,50. La mitad central se encuentra entre 5,60 y 6,00. Conservamos los 1.471 eventos señalados por la regla del IQR porque esa clasificación estadística, por sí sola, no demuestra errores.

4. **La profundidad presenta una cola hacia valores altos.** Su mediana es 33,00 km y su media 71,31 km. La mitad central se ubica entre 15,00 y 54,80 km. Los eventos más profundos elevan el promedio.

5. **La profundidad muestra poca asociación global con la magnitud.** Pearson fue 0,023 y Spearman −0,037. Evaluaremos su aporte al combinarla con la ubicación y comparar modelos con y sin esta entrada.

6. **La ubicación muestra concentraciones espaciales.** El mapa permite identificar franjas y agrupaciones de epicentros. Conservamos latitud y longitud como variables candidatas y comprobaremos su utilidad mediante evaluación temporal.

7. **Definimos la preparación inicial.** Las entradas serán `Latitude`, `Longitude` y `Depth`, y el objetivo será `Magnitude`. Estas columnas tienen cero faltantes. La división temporal asigna 18.026 registros a entrenamiento, 2.905 a validación y 2.301 a prueba.

En la siguiente etapa compararemos un árbol de decisión y un bosque aleatorio con una referencia que prediga la mediana de entrenamiento. Usaremos MAE como métrica principal, acompañada de RMSE, R² y una revisión del error por magnitud y tipo de magnitud.

El escenario de uso será estimar la magnitud reportada de un evento ya detectado, con ubicación y profundidad disponibles, dentro del alcance del catálogo estudiado.